**Data Cleanup**

In [ ]:
"""
Data we need
- set of exams to schedule
- list of timeslots
- list of rooms available, by timeslot
- set of unique student schedules
- number of students with each unique schedule
- where classes during the semester
- set of classes that need to be split across rooms

Hard Constraints:
- room capacities
- all exams scheduled


Factors to consider:
1. student overlaps
2. 3 exams in 48 hours
3. back-to-back exams
4. in-person exams finish as early as possible
5. accomodate room requests
6. correct room type

If we have time:
- minimize exams on sunday
- fairness across majors

"""

In [ ]:
import pandas as pd

# Load the CSV files
class_info = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Class_Info2023.csv')
final_exams = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Final_Exams2023.csv')
schedule = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Schedule2023.csv')
student_reg = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/StudentRegistration2023.csv')

# Process exams (E): list of CRNs from final_exams
E = final_exams['CRN'].tolist()

# Class sizes: number of students per exam (CRN)
class_sizes = student_reg.groupby('CRN').size().to_dict()

# Timeslots (T): unique (EXAM_DATE, EXAM_TIME) from final_exams
timeslots = final_exams[['EXAM_DATE', 'EXAM_TIME']].drop_duplicates()
T = [(row['EXAM_DATE'], row['EXAM_TIME']) for _, row in timeslots.iterrows()]

# Rooms (R): unique rooms from class_info where AVAILABLE_TO_SCHEDULE == 'Y'
available_rooms = class_info[class_info['AVAILABLE_TO_SCHEDULE'] == 'Y']
rooms = available_rooms[['BUILDING_CODE', 'ROOM_NUMBER']].drop_duplicates()
R = [(row['BUILDING_CODE'], row['ROOM_NUMBER']) for _, row in rooms.iterrows()]

# Capacities: dict room -> ROOM_CAPACITY
capacities = {(row['BUILDING_CODE'], row['ROOM_NUMBER']): row['ROOM_CAPACITY'] for _, row in available_rooms.iterrows()}

# Student schedules (S): dict index -> set of CRNs
student_schedules = student_reg.groupby('PERSON_IDENTIFIER')['CRN'].apply(set).to_dict()

# Unique schedules and counts (S and N)
from collections import defaultdict
schedule_counts = defaultdict(int)
schedule_to_index = {}
index = 0
for sched in student_schedules.values():
    sched_tuple = frozenset(sched)
    if sched_tuple not in schedule_to_index:
        schedule_to_index[sched_tuple] = index
        index += 1
    schedule_counts[schedule_to_index[sched_tuple]] += 1

S = {idx: set(sched) for sched, idx in schedule_to_index.items()}
N = schedule_counts

In [ ]:
import gurobipy as gp
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

m = gp.Model(env=env)

def schedule_model(E,T,R,S,N, capacities, class_sizes):
    """
    Inputs:
    E - list of exams to be scheduled
    T - list of timeslots available
    R - list of rooms available
    S - dictionary of student schedules - schedule index : set of classes
    N - dictionary of students per schedule - schedule index : number of students with that schedule
    capacities - dictionary of room capacities
    class_sizes - dictionary mapping exam to number of students

    """
    size_e = len(E)
    size_t = len(T)
    size_r = len(R)
    size_s = len(S)

    y_vars = [(e, t) for e in range(size_e) for t in range(size_t)]
    z_vars = [(e_1, e_2, t) for e_1 in range(size_e) for e_2 in range(e_1+1, size_e) for t in range(size_t)]
    x = m.addVars(range(size_e), range(size_t), range(size_r),vtype=GRB.BINARY)

    y = m.addVars(y_vars, vtype=GRB.BINARY)
    z = m.addVars(z_vars,vtype=GRB.CONTINUOUS)
    w = [1]*size_t # figure out late penalties later
    #assume only 1 student overlap between every pair of exams
    s = {(e_1, e_2): 1 for e_1 in range(size_e) for e_2 in range(e_1+1, size_e)}

    lamb = 1
    mu = 10  # penalty for 3 exams in 48 hours
    nu = 5   # penalty for back-to-back
    
    m.setObjective(gp.quicksum(w[j]*x[i, j, k] for i in range(size_e) for j in range(size_t) for k in range(size_r))+ 
                   lamb*gp.quicksum(s[(e_1, e_2)]*z[e_1, e_2, t] for e_1 in range(size_e) for e_2 in range(e_1+1, size_e) for t in range(size_t)) +
                   mu*gp.quicksum(u[t, s] for t in range(size_t - 5) for s in range(size_s)) +
                   nu*gp.quicksum(d[t, s] for t in range(size_t - 1) for s in range(size_s)),
                GRB.MINIMIZE)
    
    # CONSTRAINTS:
    m.addConstrs((gp.quicksum(x[e,t,r] for t in range(size_t) for r in range(size_r)) == 1 for e in range(size_e)), name="assign_once")

    m.addConstrs((gp.quicksum(class_sizes[e]*x[e,t,r] for e in range(size_e)) <= capacities[r] for t in range(size_t) for r in range(size_r)), name="capacity")

    m.addConstrs((y[e,t] == gp.quicksum(x[e,t,r] for r in range(size_r)) for e in range(size_e) for t in range(size_t)), name="y_def")

    # linearizing z variables
    m.addConstrs(z[e_1,e_2,t] <= y[e_1,t] for t in range(size_t) for e_1 in range(size_e) for e_2 in range(e_1+1, size_e))
    m.addConstrs(z[e_1,e_2,t] <= y[e_2,t] for t in range(size_t) for e_1 in range(size_e) for e_2 in range(e_1+1, size_e))
    m.addConstrs(z[e_1,e_2,t] >= y[e_1,t] + y[e_2,t] - 1 for t in range(size_t) for e_1 in range(size_e) for e_2 in range(e_1+1, size_e))

    # TRACKING 3 EXAMS IN 48 HOURS
    v = m.addVars(range(size_t - 5), range(size_s), vtype=GRB.INTEGER, name="v")

    m.addConstrs((v[t, s] == gp.quicksum(x[e, t+i, r] for e in S[s] for i in range(6) for r in range(size_r)) for t in range(size_t - 5) for s in range(size_s)), name="v_def")

    u = m.addVars(range(size_t-5), range(size_s), vtype=GRB.BINARY, name="u") # Indicator variable must be binary
    for t in range(size_t - 5):
        for s in range(size_s):
            m.addGenConstrIndicator(u[t, s], True, v[t, s] >= 3)
            m.addGenConstrIndicator(u[t, s], False, v[t, s] <= 2)

    # TRACKING BACK-TO-BACK EXAMS
    d = m.addVars(range(size_t - 1), range(size_s), vtype=GRB.INTEGER, name='d')

    for s in range(size_s):
        for t in range(size_t - 1):
            m.addConstr(d[t,s] <= gp.quicksum(x[e,t,r] for e in S[s] for r in range(size_r)))
            m.addConstr(d[t,s] <= gp.quicksum(x[e,t+1,r] for e in S[s] for r in range(size_r)))
            m.addConstr(1 + d[t,s] >= gp.quicksum(x[e,t,r] + x[e,t+1,r] for e in S[s] for r in range(size_r)))

m.optimize()

print("The optimal solution is {}".format(m.x))
print("The total cost is {}".format(m.ObjVal))